# Test: load TopK residual-stream SAE as circuit-tracer transcoder (Caminho A)

**Hypothesis**: circuit-tracer's `SingleLayerTranscoder` accepts arbitrary input/output hooks. If we set both to `hook_resid_post.18`, it should function as a residual-stream SAE (input and output spaces are identical, so the "transcoder" is just our SAE's encode→decode loop).

**Test**:
1. Install circuit-tracer + nnsight backend (TransformerLens doesn't support Qwen3.5)
2. Load our SAE and convert format to circuit-tracer expectations
3. Build config.yaml with `activation: topk`, `feature_input_hook = feature_output_hook = hook_resid_post.18`
4. Try running an attribution graph on a GSM8K question

**If this works**: we can write a doc PR showing how to use residual-stream SAEs via the existing transcoder API. No code changes needed.

**If it fails**: we identify the exact error and open a PR proposing `ResidualStreamSAE` class.

Runs on RTX 6000 Blackwell 96 GB (nnsight is slower than TL but works with any HF model).

## Cell 1 — Install

In [ ]:
import sys, subprocess

def pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check)

# transformers pinned (same commit as our mechreward work)
pip('install', '-q', '--no-cache-dir',
    'git+https://github.com/huggingface/transformers.git')
pip('install', '-q',
    'huggingface_hub==1.5.0',
    'safetensors', 'einops',
    'flash-linear-attention', 'causal-conv1d')

# circuit-tracer + nnsight backend
pip('install', '-q', '--no-cache-dir',
    'circuit-tracer', 'nnsight')

# HF auth
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('✅ HF authenticated')
except Exception as e:
    print(f'HF auth skipped: {e}')

# Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/mechreward/circuit_tracer_test')
WORK.mkdir(parents=True, exist_ok=True)
print(f'Work dir: {WORK}')

## Cell 2 — Download our SAE + inspect structure

In [ ]:
import torch, json
from huggingface_hub import hf_hub_download

SAE_REPO = 'caiovicentino1/Qwen3.5-4B-SAE-L18-topk'

sae_pt = hf_hub_download(SAE_REPO, 'sae_final.pt')
sae_json = hf_hub_download(SAE_REPO, 'sae_final.json')

state = torch.load(sae_pt, map_location='cpu', weights_only=False)
if hasattr(state, 'state_dict'):
    state = state.state_dict()

with open(sae_json) as f:
    cfg = json.load(f)

print('Config fields:', list(cfg.keys()))
for k, v in cfg.items():
    print(f'  {k}: {v}')

print('\nState dict shapes:')
for k, v in state.items():
    print(f'  {k}: {tuple(v.shape) if hasattr(v, "shape") else v}')

## Cell 3 — Inspect circuit-tracer's SingleLayerTranscoder API

In [ ]:
from circuit_tracer.transcoder.single_layer_transcoder import SingleLayerTranscoder
import inspect

print('Signature:')
print(inspect.signature(SingleLayerTranscoder.__init__))
print('\nDocstring:')
print(SingleLayerTranscoder.__init__.__doc__ or '(no docstring)')

print('\nSource (first 80 lines):')
src = inspect.getsource(SingleLayerTranscoder)
for line in src.split('\n')[:80]:
    print(line)

## Cell 4 — Convert our SAE to circuit-tracer format

In [ ]:
# circuit-tracer expects weights named like W_enc, W_dec, b_enc, b_dec with specific shapes
# Our SAE already uses those names — check if shapes match what they expect

W_enc = state['W_enc']  # we stored as [d_model, d_sae]
W_dec = state['W_dec']
b_enc = state.get('b_enc', torch.zeros(W_enc.shape[1]))
b_dec = state.get('b_dec', torch.zeros(W_enc.shape[0]))

print(f'W_enc: {tuple(W_enc.shape)}  (expected: [d_model={cfg["d_model"]}, d_sae={cfg["d_sae"]}])')
print(f'W_dec: {tuple(W_dec.shape)}  (expected: [d_sae, d_model])')
print(f'b_enc: {tuple(b_enc.shape)}')
print(f'b_dec: {tuple(b_dec.shape)}')

# Circuit-tracer convention may expect transposed weights — we test both shapes later.
# Save a candidate config.yaml that works with the library
import yaml
ct_cfg = {
    'activation': 'topk',  # per PR #94
    'k': cfg['k'],
    'd_model': cfg['d_model'],
    'd_features': cfg['d_sae'],
    'layer': cfg['layer'],
    # Both input and output on residual stream post-L18 — this is the key test
    'feature_input_hook': f'blocks.{cfg["layer"]}.hook_resid_post',
    'feature_output_hook': f'blocks.{cfg["layer"]}.hook_resid_post',
    'skip_connection': True,  # so the transcoder doesn't modify residual, just reads features
    'source_model': cfg['model'],
}

cfg_path = WORK / 'config.yaml'
with open(cfg_path, 'w') as f:
    yaml.safe_dump(ct_cfg, f)
print(f'\nWrote: {cfg_path}')
print(yaml.safe_dump(ct_cfg))

# Save weights in circuit-tracer's expected layout
# (typical: transcoder_weights.pt with keys W_enc, W_dec, b_enc, b_dec)
weights_path = WORK / 'transcoder_weights.pt'
torch.save({'W_enc': W_enc, 'W_dec': W_dec, 'b_enc': b_enc, 'b_dec': b_dec}, weights_path)
print(f'Wrote: {weights_path}')

## Cell 5 — Try to instantiate SingleLayerTranscoder with our weights

In [ ]:
# Try to load via their standard loader
from circuit_tracer.transcoder.single_layer_transcoder import SingleLayerTranscoder

try:
    # Try direct instantiation first
    t = SingleLayerTranscoder(
        d_model=cfg['d_model'],
        d_features=cfg['d_sae'],
        activation='topk',
        k=cfg['k'],
    )
    print('✅ Constructor accepted')
except Exception as e:
    print(f'❌ Constructor failed: {type(e).__name__}: {e}')
    print('\nFallback: inspect expected signature')
    import inspect
    print(inspect.signature(SingleLayerTranscoder))
    raise

## Cell 6 — Load weights into the transcoder + sanity test

In [ ]:
# Load our weights into the transcoder instance
try:
    # Circuit-tracer may expect specific internal param names
    # Let's inspect what it has
    print('Transcoder parameters:')
    for name, p in t.named_parameters():
        print(f'  {name}: {tuple(p.shape)}')

    # Try loading our weights, potentially with transpose
    t.W_enc.data = W_enc if W_enc.shape == t.W_enc.shape else W_enc.t()
    t.W_dec.data = W_dec if W_dec.shape == t.W_dec.shape else W_dec.t()
    t.b_enc.data = b_enc
    t.b_dec.data = b_dec
    print('✅ Weights loaded')

    # Sanity: encode → decode some random input, check reconstruction is reasonable
    h = torch.randn(1, 10, cfg['d_model'])
    with torch.no_grad():
        feats = t.encode(h) if hasattr(t, 'encode') else None
        recon = t.decode(feats) if feats is not None and hasattr(t, 'decode') else None
    if recon is not None:
        mse = (recon - h).pow(2).mean().item()
        print(f'✅ Forward pass OK. Random-input recon MSE: {mse:.4f} (random should be high, real data low)')
    else:
        print('⚠️  No encode/decode methods — API may be different. Inspect class.')
        print([m for m in dir(t) if not m.startswith('_')])
except Exception as e:
    print(f'❌ Failed: {type(e).__name__}: {e}')
    raise

## Cell 7 — End-to-end: attribution graph on a GSM8K question

If we get here, the format works. Now try the real thing: load Qwen3.5-4B via nnsight backend + attribute a generation to features.

**Warning**: nnsight + 4B model + attribution is memory-heavy. May need to use a smaller test question.

In [ ]:
# This is the real test. If this produces a valid attribution graph, Caminho A works.
from circuit_tracer import ReplacementModel, attribute

# Load the model via nnsight (since TL doesn't support Qwen3.5)
try:
    rmodel = ReplacementModel.from_pretrained(
        'Qwen/Qwen3.5-4B',
        transcoders={cfg['layer']: t},  # our single-layer residual SAE at L18
        backend='nnsight',
        dtype=torch.bfloat16,
    )
    print('✅ ReplacementModel loaded')
except Exception as e:
    print(f'❌ ReplacementModel failed: {type(e).__name__}: {e}')
    raise

# Attribution test
prompt = "Q: What is 12 * 5?\nA: Let's think step by step."
try:
    graph = attribute(
        rmodel, prompt,
        max_new_tokens=32,
        sparsity_threshold=0.01,
    )
    print('✅ Attribution graph generated')
    print(f'Graph nodes: {len(graph.nodes)}  edges: {len(graph.edges)}')
except Exception as e:
    print(f'❌ Attribution failed: {type(e).__name__}: {e}')
    raise

## Cell 8 — Summary & next steps

Based on which cell above failed, decide:

| Failed at | Implication | Next step |
|---|---|---|
| Cell 5 (constructor) | API needs different args | Adapt constructor call, possibly file an issue |
| Cell 6 (weight load) | Our tensor shapes/names differ | Write a format converter (easy PR) |
| Cell 7 (ReplacementModel) | Res-stream hook not supported as transcoder | This IS the gap — big PR needed |
| All pass | Caminho A works out of the box | Write doc PR showing how |

If all 7 cells pass, next step is visualizing the graph on Neuronpedia or locally (their React frontend).